In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from PIL import Image

## Bố trí Dữ liệu (Data Restructuring)

Đối với Module Meat (Sử dụng Custom Dataset):
Vì ảnh nằm chung trong train/ và phân biệt bằng tiền tố, thay vì tốn công di chuyển file bằng code, chúng ta sẽ viết một class CustomDataset để tự động đọc tiền tố và gán nhãn.

FRESH_... -> Nhãn 0

HALF-FRESH_... -> Nhãn 1

SPOILED_... -> Nhãn 2

Đối với Module Fruit:
Dữ liệu đang ở dạng apple/fresh/, apple/rotten/. Bạn cần gom tất cả fresh của các loại quả vào chung một thư mục Fresh/, và rotten vào thư mục Rotten/

In [ ]:
def restructure_fruit_data(source_dir, dest_dir):
    classes = ['fresh', 'rotten']
    for cls in classes:
        os.makedirs(os.path.join(dest_dir, 'train', cls), exist_ok=True)
        
    for fruit in os.listdir(source_dir):
        fruit_path = os.path.join(source_dir, fruit)
        if os.path.isdir(fruit_path):
            for cls in classes:
                cls_path = os.path.join(fruit_path, cls)
                if os.path.exists(cls_path):
                    for img in os.listdir(cls_path):
                        src_img = os.path.join(cls_path, img)
                        # Đổi tên file để tránh trùng lặp giữa táo và chuối
                        dst_img = os.path.join(dest_dir, 'train', cls, f"{fruit}_{img}")
                        shutil.copy(src_img, dst_img)
restructure_fruit_data('/kaggle/input/fruit-data', '/kaggle/working/fruit_dataset')

## Data Augmentation

In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Mô phỏng ánh sáng tủ lạnh/nilon
    transforms.ToTensor(),
    normalize,
])

valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

## Dataset & Loader cho Meat

In [ ]:
class MeatDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        if img_name.startswith('FRESH'):
            label = 0
        elif img_name.startswith('HALF-FRESH'):
            label = 1
        else: # SPOILED
            label = 2
            
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Khởi tạo DataLoader
meat_train_data = MeatDataset('/kaggle/input/meat/train', transform=train_transforms)
train_loader = DataLoader(meat_train_data, batch_size=32, shuffle=True)

## Cấu hình Transfer Learning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def create_model(num_classes):
    # Load ResNet50 pretrained
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    # 1. Đóng băng toàn bộ mạng ban đầu
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. Mở khóa (Unfreeze) block cuối cùng (layer4) để Finetune
    for param in model.layer4.parameters():
        param.requires_grad = True
        
    # 3. Thay đổi lớp FC (Fully Connected) cho số class của dự án
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5), # Thêm Dropout để giảm Overfitting
        nn.Linear(num_ftrs, num_classes)
    )
    
    return model.to(device)

# Model thịt: 3 classes, Model trái cây: 2 classes
model_meat = create_model(num_classes=3)

## Chạy huấn luyện

In [ ]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs, checkpoint_path='checkpoint.pth'):
    best_loss = float('inf')
    
    for epoch in range(num_epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = 100 * correct / total
        
        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        val_epoch_loss = val_loss / len(valid_loader.dataset)
        val_epoch_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%")
        print(f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}%")
        
        # --- LƯU CHECKPOINT ĐỀ PHÒNG CRASH ---
        if val_epoch_loss < best_loss:
            best_loss = val_epoch_loss
            print("=> Validation loss giảm, đang lưu checkpoint...")
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_loss': best_loss
            }
            torch.save(checkpoint, checkpoint_path)
            
            # Lưu riêng thêm một bản weights nhẹ chỉ để deploy
            torch.save(model.state_dict(), checkpoint_path.replace('.pth', '_weights_only.pth'))
        print("-" * 30)

# Khởi chạy
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_meat.parameters()), lr=1e-4)
train_model(model_meat, train_loader, valid_loader, criterion, optimizer, num_epochs=15, checkpoint_path='/kaggle/working/meat_model_ckpt.pth')